In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForDepthEstimation, AutoConfig, AutoImageProcessor
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import json
from tqdm import tqdm
import pandas as pd
import os
from typing import Dict, List, Optional, Tuple
import logging
import traceback
import sys
import torchvision.transforms as transforms
import torch.nn.functional as F  # This is for loss functions
import torchvision.transforms.functional as TF  # This is for image transforms

In [4]:
# Set up logging to help us understand what's happening during dataset loading
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [5]:

class ResizeWithPad:
    """
    A custom transform that handles images of any size and ensures they all end up
    exactly the same dimensions. This is particularly important for images like
    your 118x256 front.jpg.
    """
    def __init__(self, target_size):
        self.target_size = target_size
        logger.info(f"Initializing ResizeWithPad with target size {target_size}")
    
    def __call__(self, img):
        # First, ensure we're working with a PIL Image
        if isinstance(img, torch.Tensor):
            img = TF.to_pil_image(img)
        
        # Get current dimensions
        original_width, original_height = img.size
        target_height, target_width = self.target_size
        
        # Log original dimensions for debugging
        logger.debug(f"Processing image: {original_width}x{original_height} -> {target_width}x{target_height}")
        
        # Calculate aspect ratios
        orig_aspect = original_width / original_height
        target_aspect = target_width / target_height
        
        # Determine scaling approach
        if orig_aspect > target_aspect:
            # Image is wider than target aspect ratio
            scale_factor = target_width / original_width
            new_width = target_width
            new_height = int(original_height * scale_factor)
        else:
            # Image is taller than target aspect ratio
            scale_factor = target_height / original_height
            new_height = target_height
            new_width = int(original_width * scale_factor)
        
        # Resize image while maintaining aspect ratio
        resized_img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        # Create new image with padding
        result = Image.new('RGB', (target_width, target_height), (0, 0, 0))
        
        # Calculate padding to center the image
        pad_x = (target_width - new_width) // 2
        pad_y = (target_height - new_height) // 2
        
        # Paste resized image onto padded background
        result.paste(resized_img, (pad_x, pad_y))
        
        # Convert to tensor
        result_tensor = TF.to_tensor(result)
        
        # Log final dimensions
        logger.debug(f"Final tensor shape: {result_tensor.shape}")
        
        return result_tensor


In [6]:
class ProductViewDataset(Dataset):
    """
    Dataset class for loading product views from product-specific directories.
    Structure: dataset/product_name/{front,back,top}.{jpg,png,...}
    """
    def __init__(self, dataset_path="../datasets", image_processor=None):
        self.dataset_path = Path(dataset_path)
        self.image_processor = image_processor
        self.image_extensions = ['.jpg', '.jpeg', '.png']  # Add more if needed
        self.product_paths = self._get_product_paths()
        print(f"Found {len(self.product_paths)} products with complete views")
        
        # Print some example paths to verify
        if len(self.product_paths) > 0:
            print("\nExample product paths:")
            print(f"Product ID: {self.product_paths[0]['id']}")
            for view in ['front', 'back', 'top']:
                print(f"{view}: {self.product_paths[0][view]}")
    
    def _find_view_file(self, product_dir: Path, view_name: str) -> Optional[Path]:
        """Find image file for a specific view, trying different extensions"""
        for ext in self.image_extensions:
            # Try lowercase
            path = product_dir / f"{view_name}{ext}"
            if path.exists():
                return path
            # Try uppercase
            path = product_dir / f"{view_name.upper()}{ext}"
            if path.exists():
                return path
        return None
    
    def _get_product_paths(self):
        """Collect all products that have all three views"""
        products = []
        
        # Print total number of directories found
        all_dirs = list(self.dataset_path.iterdir())
        print(f"\nFound {len(all_dirs)} total directories in dataset path")
        
        for product_dir in all_dirs:
            if not product_dir.is_dir():
                continue
                
            # Find view files
            views = {
                'front': self._find_view_file(product_dir, 'front'),
                'back': self._find_view_file(product_dir, 'back'),
                'top': self._find_view_file(product_dir, 'top')
            }
            
            # Check if all views exist
            if all(views.values()):
                products.append({
                    'id': product_dir.name,
                    'front': str(views['front']),
                    'back': str(views['back']),
                    'top': str(views['top'])
                })
            else:
                # Print missing views for debugging
                missing = [view for view, path in views.items() if path is None]
                print(f"Skipping {product_dir.name} - missing views: {missing}")
        
        return products
    
    def __len__(self):
        """Return the number of products in the dataset"""
        return len(self.product_paths)
    
    def __getitem__(self, idx):
        """Get a product's views, ensuring consistent sizing"""
        product = self.product_paths[idx]
        
        # Load and process images
        images = {}
        for view in ['front', 'back', 'top']:
            try:
                img = Image.open(product[view]).convert('RGB')
                if self.image_processor:
                    img = self.image_processor(img, return_tensors="pt")['pixel_values'][0]
                else:
                    transform = transforms.Compose([
                        transforms.Resize(256),
                        transforms.CenterCrop(256),
                        transforms.ToTensor(),
                    ])
                    img = transform(img)
                images[view] = img
            except Exception as e:
                print(f"Error loading {view} view for product {product['id']}: {str(e)}")
                raise
        
        return images

def verify_dataset_dimensions(dataset_path="../datasets"):
    """
    Test function to verify dataset loading
    """
    print("Testing dataset loading...")
    print(f"Dataset path: {dataset_path}")
    
    # List contents of dataset directory
    dataset_dir = Path(dataset_path)
    if not dataset_dir.exists():
        print(f"Error: Dataset directory {dataset_path} does not exist!")
        return
    
    print("\nContents of dataset directory:")
    for item in dataset_dir.iterdir():
        if item.is_dir():
            print(f"\nProduct directory: {item.name}")
            print("Files:")
            for file in item.iterdir():
                print(f"  - {file.name}")
    
    # Try creating dataset
    dataset = ProductViewDataset(dataset_path)
    
    if len(dataset) == 0:
        print("\nNo valid products found! Please check:")
        print("1. Product directories contain all three views (front, back, top)")
        print("2. Image files are named correctly (front.jpg, back.jpg, top.jpg)")
        print("3. Image files are in a supported format (.jpg, .jpeg, .png)")
    else:
        print(f"\nSuccessfully loaded {len(dataset)} products!")
        print("\nVerifying dimensions for all products...")
        for i in range(len(dataset)):
            try:
                sample = dataset[i]
                if i == 0:  # Print details for first product
                    print("\nFirst product details:")
                    for view, tensor in sample.items():
                        print(f"{view} shape: {tensor.shape}")
            except Exception as e:
                print(f"Error processing product {i}: {str(e)}")

In [7]:
verify_dataset_dimensions()

Testing dataset loading...
Dataset path: ../datasets

Contents of dataset directory:

Product directory: product_100
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_1056
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_1088
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_114
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_1149
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_1210
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_1211
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_1215
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_1218
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_1235
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_1240
Files:
  - back.jpg
  - front.jpg
  - top.jpg

Product directory: product_1243
Files:
  - back

In [53]:
class DepthFineTuner:
    def __init__(self, model_name="depth-anything/Depth-Anything-V2-base-hf"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Using device: {self.device}")
        
        # Initialize model and processor
        self.config = AutoConfig.from_pretrained(model_name)
        self.model = AutoModelForDepthEstimation.from_pretrained(
            model_name,
            config=self.config
        ).to(self.device)
        
        # Training components
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-5)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', patience=2, factor=0.5
        )
        
        # Initialize training history
        self.history = {
            'geometric_loss': [],
            'smoothness_loss': [],
            'edge_loss': [],
            'total_loss': []
        }
    
    def geometric_consistency_loss(self, front_depth, top_depth):
        """Compute geometric consistency between views using torch.nn.functional"""
        try:
            # Extract height information from top view
            top_height = torch.mean(top_depth, dim=2)
            front_depth_profile = torch.mean(front_depth, dim=2)
            
            # Normalize
            top_height = (top_height - top_height.min()) / (top_height.max() - top_height.min() + 1e-6)
            front_depth_profile = (front_depth_profile - front_depth_profile.min()) / (front_depth_profile.max() - front_depth_profile.min() + 1e-6)
            
            return F.mse_loss(front_depth_profile, top_height)
        except Exception as e:
            logger.error(f"Error in geometric consistency loss: {str(e)}")
            raise
    
    def smoothness_loss(self, depth):
        """Compute smoothness loss using gradients"""
        try:
            # Compute gradients
            grad_x = torch.abs(depth[:, :, 1:] - depth[:, :, :-1])
            grad_y = torch.abs(depth[:, 1:, :] - depth[:, :-1, :])
            
            return torch.mean(grad_x) + torch.mean(grad_y)
        except Exception as e:
            logger.error(f"Error in smoothness loss: {str(e)}")
            return torch.tensor(0.0, device=self.device)
    
    def edge_consistency_loss(self, front_depth, back_depth):
        """Compute edge consistency loss"""
        try:
            # Simple edge detection using gradients
            front_edges = self._detect_edges(front_depth)
            back_edges = self._detect_edges(torch.flip(back_depth, [2]))
            
            return F.mse_loss(front_edges, back_edges)
        except Exception as e:
            logger.error(f"Error in edge consistency loss: {str(e)}")
            return torch.tensor(0.0, device=self.device)
    
    def _detect_edges(self, depth):
        """Helper function to detect edges in depth maps"""
        grad_x = torch.abs(depth[:, :, 1:] - depth[:, :, :-1])
        grad_y = torch.abs(depth[:, 1:, :] - depth[:, :-1, :])
        
        # Pad to maintain size
        grad_x = F.pad(grad_x, (0, 1), mode='replicate')
        grad_y = F.pad(grad_y, (0, 0, 0, 1), mode='replicate')
        
        return torch.sqrt(grad_x**2 + grad_y**2)
    
    def train_step(self, batch):
        """Perform one training step with error handling"""
        try:
            self.model.train()
            self.optimizer.zero_grad()
            
            # Move images to device
            for view in batch:
                batch[view] = batch[view].to(self.device)
            
            # Get depth predictions
            front_depth = self.model(batch['front']).predicted_depth
            back_depth = self.model(batch['back']).predicted_depth
            top_depth = self.model(batch['top']).predicted_depth
            
            # Compute losses
            geo_loss = self.geometric_consistency_loss(front_depth, top_depth)
            smooth_loss = self.smoothness_loss(front_depth) + self.smoothness_loss(back_depth)
            edge_loss = self.edge_consistency_loss(front_depth, back_depth)
            
            # Total loss
            total_loss = geo_loss + 0.1 * smooth_loss + 0.1 * edge_loss
            
            # Backpropagate
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            return {
                'geometric_loss': geo_loss.item(),
                'smoothness_loss': smooth_loss.item(),
                'edge_loss': edge_loss.item(),
                'total_loss': total_loss.item()
            }
        except Exception as e:
            logger.error(f"Error in training step: {str(e)}")
            raise
    
    def fine_tune(self, train_dataset, batch_size=2, num_epochs=10):
        """Fine-tune the model with proper error handling"""
        logger.info(f"Starting fine-tuning for {num_epochs} epochs")
        logger.info(f"Training on {len(train_dataset)} products")
        
        dataloader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=True
        )
        
        num_batches = len(dataloader)
        logger.info(f"Created DataLoader with {num_batches} batches")
        
        for epoch in range(num_epochs):
            epoch_losses = []
            progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
            
            for batch_idx, batch in enumerate(progress_bar):
                try:
                    losses = self.train_step(batch)
                    epoch_losses.append(losses)
                    
                    # Update progress bar
                    progress_bar.set_postfix({
                        k: f"{v:.4f}" for k, v in losses.items()
                    })
                    
                except Exception as e:
                    logger.error(f"Error processing batch {batch_idx}: {str(e)}")
                    continue
            
            # Compute epoch summary only if we have valid losses
            if epoch_losses:
                avg_losses = {
                    k: np.mean([x[k] for x in epoch_losses])
                    for k in epoch_losses[0].keys()
                }
                
                logger.info(f"\nEpoch {epoch+1} Summary:")
                for loss_name, value in avg_losses.items():
                    logger.info(f"{loss_name}: {value:.4f}")
                
                # Update learning rate
                self.scheduler.step(avg_losses['total_loss'])
                
                # Save checkpoint
                self.save_checkpoint(f'checkpoint_epoch_{epoch+1}.pt')
            else:
                logger.warning(f"No valid losses for epoch {epoch+1}")
    
    def save_checkpoint(self, filename):
        """Save training checkpoint"""
        try:
            checkpoint = {
                'model_state_dict': self.model.state_dict(),
                'optimizer_state_dict': self.optimizer.state_dict(),
                'scheduler_state_dict': self.scheduler.state_dict(),
                'history': self.history
            }
            torch.save(checkpoint, filename)
            logger.info(f"Saved checkpoint: {filename}")
        except Exception as e:
            logger.error(f"Error saving checkpoint: {str(e)}")
    
    def save_model(self, path='fine_tuned_depth_estimation'):
        """Save the fine-tuned model"""
        try:
            self.model.save_pretrained(path)
            logger.info(f"Model saved to {path}")
        except Exception as e:
            logger.error(f"Error saving model: {str(e)}")

In [54]:
# Initialize components
try:
    fine_tuner = DepthFineTuner()
    logger.info("DepthFineTuner initialized successfully")
    
    # In your training script
    dataset = ProductViewDataset("../datasets")

    logger.info(f"Dataset loaded with {len(dataset)} products")
    
    # Fine-tune model
    fine_tuner.fine_tune(dataset)
    
    # Save final model
    fine_tuner.save_model('fine_tuned_depth_estimation')
    logger.info("Training completed successfully")
    
except Exception as e:
    logger.error(f"Error in main execution: {str(e)}")
    raise

INFO:__main__:Using device: cuda
INFO:__main__:DepthFineTuner initialized successfully
INFO:__main__:Dataset loaded with 169 products
INFO:__main__:Starting fine-tuning for 10 epochs
INFO:__main__:Training on 169 products
INFO:__main__:Created DataLoader with 85 batches



Found 169 total directories in dataset path
Found 169 products with complete views

Example product paths:
Product ID: product_100
front: ..\datasets\product_100\front.jpg
back: ..\datasets\product_100\back.jpg
top: ..\datasets\product_100\top.jpg


Epoch 1/10: 100%|██████████| 85/85 [01:27<00:00,  1.03s/it, geometric_loss=0.0001, smoothness_loss=0.0045, edge_loss=0.0000, total_loss=0.0005]
INFO:__main__:
Epoch 1 Summary:
INFO:__main__:geometric_loss: 0.0143
INFO:__main__:smoothness_loss: 0.0351
INFO:__main__:edge_loss: 0.0055
INFO:__main__:total_loss: 0.0183
INFO:__main__:Saved checkpoint: checkpoint_epoch_1.pt
Epoch 2/10: 100%|██████████| 85/85 [01:37<00:00,  1.15s/it, geometric_loss=0.0000, smoothness_loss=0.0014, edge_loss=0.0000, total_loss=0.0001]
INFO:__main__:
Epoch 2 Summary:
INFO:__main__:geometric_loss: 0.0000
INFO:__main__:smoothness_loss: 0.0013
INFO:__main__:edge_loss: 0.0000
INFO:__main__:total_loss: 0.0002
INFO:__main__:Saved checkpoint: checkpoint_epoch_2.pt
Epoch 3/10: 100%|██████████| 85/85 [01:45<00:00,  1.24s/it, geometric_loss=0.0000, smoothness_loss=0.0005, edge_loss=0.0000, total_loss=0.0000]
INFO:__main__:
Epoch 3 Summary:
INFO:__main__:geometric_loss: 0.0000
INFO:__main__:smoothness_loss: 0.0005
INFO:__ma

In [1]:
class DepthModelEvaluator:
    def __init__(self, original_model, fine_tuned_model, dataset):
        self.original_model = original_model
        self.fine_tuned_model = fine_tuned_model
        self.dataset = dataset
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
    def evaluate_depth_consistency(self, sample):
        """
        Compare depth predictions between original and fine-tuned models.
        This helps us understand if the fine-tuning improved view consistency.
        """
        with torch.no_grad():
            # Get predictions from both models
            original_front = self.original_model(sample['front'].to(self.device)).predicted_depth
            original_top = self.original_model(sample['top'].to(self.device)).predicted_depth
            
            fine_tuned_front = self.fine_tuned_model(sample['front'].to(self.device)).predicted_depth
            fine_tuned_top = self.fine_tuned_model(sample['top'].to(self.device)).predicted_depth
            
            # Create visualization
            fig, axes = plt.subplots(2, 2, figsize=(15, 15))
            
            # Original model predictions
            im1 = axes[0,0].imshow(original_front.squeeze().cpu(), cmap='plasma')
            axes[0,0].set_title('Original Model - Front Depth')
            plt.colorbar(im1, ax=axes[0,0])
            
            im2 = axes[0,1].imshow(original_top.squeeze().cpu(), cmap='plasma')
            axes[0,1].set_title('Original Model - Top Depth')
            plt.colorbar(im2, ax=axes[0,1])
            
            # Fine-tuned model predictions
            im3 = axes[1,0].imshow(fine_tuned_front.squeeze().cpu(), cmap='plasma')
            axes[1,0].set_title('Fine-tuned Model - Front Depth')
            plt.colorbar(im3, ax=axes[1,0])
            
            im4 = axes[1,1].imshow(fine_tuned_top.squeeze().cpu(), cmap='plasma')
            axes[1,1].set_title('Fine-tuned Model - Top Depth')
            plt.colorbar(im4, ax=axes[1,1])
            
            plt.tight_layout()
            return fig
    
    def evaluate_edge_preservation(self, sample):
        """
        Compare edge detection between models.
        This shows if the fine-tuned model better preserves product edges.
        """
        def detect_edges(depth_map):
            grad_x = torch.abs(depth_map[:, :, 1:] - depth_map[:, :, :-1])
            grad_y = torch.abs(depth_map[:, 1:, :] - depth_map[:, :-1, :])
            return torch.sqrt(F.pad(grad_x, (0,1))**2 + F.pad(grad_y, (0,0,0,1))**2)
        
        with torch.no_grad():
            # Get depth predictions
            orig_depth = self.original_model(sample['front'].to(self.device)).predicted_depth
            fine_tuned_depth = self.fine_tuned_model(sample['front'].to(self.device)).predicted_depth
            
            # Detect edges
            orig_edges = detect_edges(orig_depth)
            fine_tuned_edges = detect_edges(fine_tuned_depth)
            
            # Visualize
            fig, axes = plt.subplots(1, 2, figsize=(15, 7))
            axes[0].imshow(orig_edges.squeeze().cpu(), cmap='magma')
            axes[0].set_title('Original Model Edges')
            axes[1].imshow(fine_tuned_edges.squeeze().cpu(), cmap='magma')
            axes[1].set_title('Fine-tuned Model Edges')
            
            return fig

    def evaluate_surface_smoothness(self, sample):
        """
        Compare surface smoothness between models.
        Shows if the fine-tuned model produces more natural-looking surfaces.
        """
        def compute_smoothness(depth_map):
            grad_x = torch.abs(depth_map[:, :, 1:] - depth_map[:, :, :-1])
            grad_y = torch.abs(depth_map[:, 1:, :] - depth_map[:, :-1, :])
            return torch.mean(grad_x) + torch.mean(grad_y)
        
        results = {}
        with torch.no_grad():
            for view in ['front', 'back', 'top']:
                orig_depth = self.original_model(sample[view].to(self.device)).predicted_depth
                fine_tuned_depth = self.fine_tuned_model(sample[view].to(self.device)).predicted_depth
                
                results[f'{view}_original'] = compute_smoothness(orig_depth).item()
                results[f'{view}_fine_tuned'] = compute_smoothness(fine_tuned_depth).item()
        
        return results

In [ ]:
# Save the processor configuration
processor = AutoImageProcessor.from_pretrained("depth-anything/Depth-Anything-V2-base-hf")
processor.save_pretrained("fine_tuned_depth_estimation/")

['fine_tuned_depth_estimation/preprocessor_config.json']